# Part 3: Data Analytics with PySpark

**Objective:** Analyze BLS time-series data and US population data using PySpark.

**Three Queries:**
1. **Query 1:** Mean and standard deviation of US population for years 2013-2018
2. **Query 2:** For each series_id, find the best year (year with maximum sum of quarterly values)
3. **Query 3:** Join BLS data with population for series_id=PRS30006032, period=Q01

**Data Sources:**
- BLS: `../data/bls/pr.data.0.Current` (fixed-width text file)
- Population: `../data/population.json` (JSON from DataUSA API)

## Section 1: Initialize Spark Session and Config

In [3]:
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as spark_sum, mean, stddev, 
    when, cast, round as spark_round, trim, 
    min, max, row_number, lit
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    DoubleType, LongType
)
from pyspark.sql.window import Window
import json

# Create Spark Session with configurations
spark = SparkSession.builder \
    .appName("Rearc-DataQuest-Part3") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("✓ Spark Session created")
print(f"  App Name: {spark.sparkContext.appName}")
print(f"  Spark Version: {spark.version}")

✓ Spark Session created
  App Name: Rearc-DataQuest-Part3
  Spark Version: 3.5.1


## Section 2: Load Input Data into Spark DataFrames

### Load BLS Time-Series Data
The BLS file `pr.data.0.Current` is a fixed-width text file with columns:
- `series_id` (12 chars)
- `year` (4 digits)
- `period` (3 chars, e.g., Q01)
- `value` (numeric)

In [28]:
# Load BLS Data (tab-delimited file with whitespace-trimmed columns)
bls_path = "../data/bls/pr.data.0.Current"

# Read as CSV with tab delimiter and trim column names
raw_df = spark.read \
    .option("delimiter", "\t") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(bls_path)

# Rename columns to remove leading/trailing spaces
col_mapping = {old: old.strip() for old in raw_df.columns}
bls_df = raw_df.select([col(old).alias(col_mapping[old]) for old in raw_df.columns])

# Select relevant columns, trim string values, and filter out nulls
bls_df = bls_df.select(
    trim(col("series_id")).alias("series_id"),
    col("year"),
    trim(col("period")).alias("period"),
    col("value")
).filter(col("year").isNotNull() & col("value").isNotNull()) \
 .withColumn("year", col("year").cast(IntegerType())) \
 .withColumn("value", col("value").cast(DoubleType()))

print(f"✓ BLS Data Loaded")
print(f"  Total records: {bls_df.count()}")
print(f"  Schema:")
bls_df.printSchema()
print(f"\n  Sample rows:")
bls_df.limit(5).show(truncate=False)

✓ BLS Data Loaded
  Total records: 38232
  Schema:
root
 |-- series_id: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- period: string (nullable = true)
 |-- value: double (nullable = true)


  Sample rows:
+-----------+----+------+-----+
|series_id  |year|period|value|
+-----------+----+------+-----+
|PRS30006011|1995|Q01   |2.6  |
|PRS30006011|1995|Q02   |2.1  |
|PRS30006011|1995|Q03   |0.9  |
|PRS30006011|1995|Q04   |0.1  |
|PRS30006011|1995|Q05   |1.4  |
+-----------+----+------+-----+



### Load Population Data (from JSON)

In [5]:
# Load Population Data (JSON file from Part 2)
pop_path = "../data/population.json"

# Read JSON and extract the 'data' array
with open(pop_path, 'r') as f:
    pop_json = json.load(f)

# Create DataFrame from the 'data' array
pop_records = pop_json.get('data', [])
pop_df = spark.createDataFrame(pop_records)

# Rename columns for clarity and cast types
pop_df = pop_df.select(
    col("Year").cast(IntegerType()).alias("year"),
    col("Population").cast(LongType()).alias("population")
).distinct()

print(f"✓ Population Data Loaded")
print(f"  Total records: {pop_df.count()}")
print(f"  Schema:")
pop_df.printSchema()
print(f"\n  Sample rows:")
pop_df.orderBy("year").show()

✓ Population Data Loaded


  Total records: 11
  Schema:
root
 |-- year: integer (nullable = true)
 |-- population: long (nullable = true)


  Sample rows:
+----+----------+
|year|population|
+----+----------+
|2013| 316128839|
|2014| 318857056|
|2015| 321418821|
|2016| 323127515|
|2017| 325719178|
|2018| 327167439|
|2019| 328239523|
|2021| 331893745|
|2022| 333287562|
|2023| 334914896|
|2024| 340110990|
+----+----------+



## Section 3 & 4: Query 1 - Population Statistics (2013-2018)

In [29]:
# Query 1: Mean and Standard Deviation of US Population (2013-2018)
pop_filtered = pop_df.filter((col("year") >= 2013) & (col("year") <= 2018))

query1_stats = pop_filtered.agg(
    mean("population").alias("mean_population"),
    stddev("population").alias("stddev_population")
)

print("=" * 70)
print("QUERY 1 RESULT: Population Statistics (2013-2018)")
print("=" * 70)
query1_stats.show(truncate=False)

# Extract values for display
q1_mean = query1_stats.collect()[0]["mean_population"]
q1_stddev = query1_stats.collect()[0]["stddev_population"]
print(f"\n✓ Mean US Population (2013-2018): {q1_mean:,.2f}")
print(f"✓ Std Dev US Population (2013-2018): {q1_stddev:,.2f}")

QUERY 1 RESULT: Population Statistics (2013-2018)
+---------------+-----------------+
|mean_population|stddev_population|
+---------------+-----------------+
|3.22069808E8   |4158441.040908092|
+---------------+-----------------+


✓ Mean US Population (2013-2018): 322,069,808.00
✓ Std Dev US Population (2013-2018): 4,158,441.04


## Section 5: Query 2 - Best Year per Series ID (Max Sum of Quarterly Values)

In [30]:
# Query 2: Best Year per Series ID (highest annual sum)
# Group by series_id and year, sum all quarterly values
annual_sums = bls_df.groupBy("series_id", "year").agg(
    spark_sum("value").alias("annual_sum")
)

# Window function to rank years within each series by annual_sum (descending)
w = Window.partitionBy("series_id").orderBy(col("annual_sum").desc())
ranked = annual_sums.withColumn("rank", row_number().over(w))

# Get only the top year per series (rank = 1)
query2_result = ranked.filter(col("rank") == 1).select(
    col("series_id"),
    col("year"),
    col("annual_sum").alias("value")
).orderBy("series_id")

print("=" * 70)
print("QUERY 2 RESULT: Best Year per Series ID (Top Annual Sum)")
print("=" * 70)
print(f"Total unique series: {query2_result.count()}")
query2_result.limit(20).show(truncate=False)

print("\n✓ Sample results:")
for row in query2_result.limit(5).collect():
    print(f"  Series {row['series_id']}: Year {row['year']} (sum={row['value']:.2f})")

QUERY 2 RESULT: Best Year per Series ID (Top Annual Sum)
Total unique series: 282
+-----------+----+------------------+
|series_id  |year|value             |
+-----------+----+------------------+
|PRS30006011|2022|20.5              |
|PRS30006012|2022|17.1              |
|PRS30006013|1998|705.895           |
|PRS30006021|2010|17.7              |
|PRS30006022|2010|12.399999999999999|
|PRS30006023|2014|503.21600000000007|
|PRS30006031|2022|20.6              |
|PRS30006032|2021|17.0              |
|PRS30006033|1998|702.672           |
|PRS30006061|2022|34.5              |
|PRS30006062|2021|29.800000000000004|
|PRS30006063|2025|665.334           |
|PRS30006081|2021|24.5              |
|PRS30006082|2021|24.5              |
|PRS30006083|2022|131.294           |
|PRS30006091|2002|43.400000000000006|
|PRS30006092|2002|44.3              |
|PRS30006093|2013|514.158           |
|PRS30006101|2020|33.1              |
|PRS30006102|2020|35.7              |
+-----------+----+------------------+


✓ Sa

## Section 4 & 5: Query 3 - Join BLS with Population (Series PRS30006032, Period Q01)

In [31]:
# Query 3: Join BLS (specific series_id & period) with Population data
# Filter BLS for the specific series and period
bls_filtered = bls_df.filter(
    (col("series_id") == "PRS30006032") & 
    (col("period") == "Q01")
)

print(f"Filtered BLS records: {bls_filtered.count()}")
print("Sample BLS filtered data:")
bls_filtered.orderBy("year").show(truncate=False)

# Left join BLS with population on year
query3_result = bls_filtered.join(
    pop_df,
    on="year",
    how="left"
).select(
    col("series_id"),
    col("year"),
    col("period"),
    col("value"),
    col("population")
).orderBy("year")

print("\n" + "=" * 70)
print("QUERY 3 RESULT: BLS (PRS30006032, Q01) joined with Population")
print("=" * 70)
query3_result.show(truncate=False)

print("\n✓ Join complete:")
print(f"  Total rows: {query3_result.count()}")
print(f"  Rows with population data: {query3_result.filter(col('population').isNotNull()).count()}")

Filtered BLS records: 32
Sample BLS filtered data:
+-----------+----+------+-----+
|series_id  |year|period|value|
+-----------+----+------+-----+
|PRS30006032|1995|Q01   |0.0  |
|PRS30006032|1996|Q01   |-4.2 |
|PRS30006032|1997|Q01   |2.8  |
|PRS30006032|1998|Q01   |0.9  |
|PRS30006032|1999|Q01   |-4.1 |
|PRS30006032|2000|Q01   |0.5  |
|PRS30006032|2001|Q01   |-6.3 |
|PRS30006032|2002|Q01   |-6.6 |
|PRS30006032|2003|Q01   |-5.7 |
|PRS30006032|2004|Q01   |2.0  |
|PRS30006032|2005|Q01   |-0.5 |
|PRS30006032|2006|Q01   |1.8  |
|PRS30006032|2007|Q01   |-0.8 |
|PRS30006032|2008|Q01   |-3.5 |
|PRS30006032|2009|Q01   |-21.0|
|PRS30006032|2010|Q01   |3.2  |
|PRS30006032|2011|Q01   |1.5  |
|PRS30006032|2012|Q01   |2.5  |
|PRS30006032|2013|Q01   |0.5  |
|PRS30006032|2014|Q01   |-0.1 |
+-----------+----+------+-----+
only showing top 20 rows


QUERY 3 RESULT: BLS (PRS30006032, Q01) joined with Population
+-----------+----+------+-----+----------+
|series_id  |year|period|value|population|
+-----

## Section 6: Validate Spark Output Against Expected Results

In [32]:
# Data Quality Checks

print("=" * 70)
print("DATA QUALITY VALIDATION")
print("=" * 70)

# Check 1: BLS data - nulls and counts
print("\n1. BLS Data Quality:")
bls_null_counts = bls_df.select(
    spark_sum(when(col("series_id").isNull(), 1).otherwise(0)).alias("series_id_nulls"),
    spark_sum(when(col("year").isNull(), 1).otherwise(0)).alias("year_nulls"),
    spark_sum(when(col("period").isNull(), 1).otherwise(0)).alias("period_nulls"),
    spark_sum(when(col("value").isNull(), 1).otherwise(0)).alias("value_nulls")
)
bls_null_counts.show()

print(f"   Unique series_ids: {bls_df.select('series_id').distinct().count()}")
min_max = bls_df.agg(min('year'), max('year')).collect()[0]
print(f"   Year range: {min_max[0]} - {min_max[1]}")
print(f"   Unique periods: {bls_df.select('period').distinct().count()}")

# Check 2: Population data - nulls and counts
print("\n2. Population Data Quality:")
pop_null_counts = pop_df.select(
    spark_sum(when(col("year").isNull(), 1).otherwise(0)).alias("year_nulls"),
    spark_sum(when(col("population").isNull(), 1).otherwise(0)).alias("population_nulls")
)
pop_null_counts.show()
min_max_pop = pop_df.agg(min('year'), max('year')).collect()[0]
print(f"   Year range: {min_max_pop[0]} - {min_max_pop[1]}")

# Check 3: Query 1 validation
print("\n3. Query 1 Validation (2013-2018 population):")
print(f"   Mean: {q1_mean:,.2f}")
print(f"   StdDev: {q1_stddev:,.2f}")
print(f"   ✓ Both values should be non-null numeric")

# Check 4: Query 2 validation
print("\n4. Query 2 Validation (Best years):")
q2_count = query2_result.count()
print(f"   Unique series with best year: {q2_count}")
print(f"   ✓ No duplicate series_ids (rank=1 only)")

# Check 5: Query 3 validation
print("\n5. Query 3 Validation (PRS30006032 Q01):")
q3_count = query3_result.count()
q3_with_pop = query3_result.filter(col('population').isNotNull()).count()
print(f"   Total rows: {q3_count}")
print(f"   Rows with population match: {q3_with_pop}")
print(f"   ✓ All rows should have series_id=PRS30006032 and period=Q01")

DATA QUALITY VALIDATION

1. BLS Data Quality:
+---------------+----------+------------+-----------+
|series_id_nulls|year_nulls|period_nulls|value_nulls|
+---------------+----------+------------+-----------+
|              0|         0|           0|          0|
+---------------+----------+------------+-----------+

   Unique series_ids: 282
   Year range: 1995 - 2026
   Unique periods: 5

2. Population Data Quality:
+----------+----------------+
|year_nulls|population_nulls|
+----------+----------------+
|         0|               0|
+----------+----------------+

   Year range: 2013 - 2024

3. Query 1 Validation (2013-2018 population):
   Mean: 322,069,808.00
   StdDev: 4,158,441.04
   ✓ Both values should be non-null numeric

4. Query 2 Validation (Best years):
   Unique series with best year: 282
   ✓ No duplicate series_ids (rank=1 only)

5. Query 3 Validation (PRS30006032 Q01):
   Total rows: 32
   Rows with population match: 11
   ✓ All rows should have series_id=PRS30006032 and 

## Section 7: Write Results to Storage

In [33]:
# Write Results to Parquet Format (can also use CSV, JSON, Delta, etc.)
import os

output_dir = "../output/results"
os.makedirs(output_dir, exist_ok=True)

print("=" * 70)
print("WRITING RESULTS TO STORAGE")
print("=" * 70)

# Query 1: Population Statistics
q1_path = f"{output_dir}/query1_population_stats"
query1_stats.coalesce(1).write.mode("overwrite").parquet(q1_path)
print(f"\n✓ Query 1 results written to: {q1_path}")

# Query 2: Best Years per Series
q2_path = f"{output_dir}/query2_best_years"
query2_result.coalesce(1).write.mode("overwrite").parquet(q2_path)
print(f"✓ Query 2 results written to: {q2_path} ({query2_result.count()} rows)")

# Query 3: Joined Data
q3_path = f"{output_dir}/query3_bls_population_join"
query3_result.coalesce(1).write.mode("overwrite").parquet(q3_path)
print(f"✓ Query 3 results written to: {q3_path} ({query3_result.count()} rows)")

# Also write as CSV for easy viewing
query1_stats.coalesce(1).write.mode("overwrite").csv(f"{output_dir}/query1_population_stats.csv", header=True)
query2_result.coalesce(1).write.mode("overwrite").csv(f"{output_dir}/query2_best_years.csv", header=True)
query3_result.coalesce(1).write.mode("overwrite").csv(f"{output_dir}/query3_bls_population_join.csv", header=True)

print(f"\n✓ CSV versions also saved in {output_dir}/")

WRITING RESULTS TO STORAGE

✓ Query 1 results written to: ../output/results/query1_population_stats
✓ Query 2 results written to: ../output/results/query2_best_years (282 rows)
✓ Query 3 results written to: ../output/results/query3_bls_population_join (32 rows)

✓ CSV versions also saved in ../output/results/


26/07/26 01:43:01 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1031692 ms exceeds timeout 120000 ms
26/07/26 01:43:01 WARN SparkContext: Killing executors is not supported by current scheduler.
26/07/26 01:43:05 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

## Summary

This notebook completed all three Part 3 analytics queries using PySpark:

1. **Query 1**: Computed mean and standard deviation of US population for 2013-2018
2. **Query 2**: Found the best year (max annual sum) for each series_id in the BLS dataset
3. **Query 3**: Joined BLS data (series PRS30006032, period Q01) with population data

All results have been:
- ✓ Validated for data quality
- ✓ Written to Parquet format (for downstream processing)
- ✓ Exported to CSV (for manual review)